# RAG with Google gemini


### prompt used
```
You are a Python developer building a RAG-based clinical report
generator using the OpenAI API.

## Context
I have a JSON file `results.json` — a list of patient records.
Each record has the following structure:

{
  "filename":         "rep1.pdf",
  "patient_name":     "John Doe4",
  "calcium":          "9.7 mg/dl",
  "calcium_range":    "8.7 - 10.4",
  "potassium":        "5.03 mmol/L",
  "potassium_range":  "3.5 - 5.1",
  "sodium":           "145.0 mmol/L",
  "sodium_range":     "136 - 145",
  "comments": [
    "Calcium: 9.7 mg/dl is NORMAL (normal range: 8.7 - 10.4)",
    "Potassium: 5.03 mmol/L is NORMAL (normal range: 3.5 - 5.1)",
    "Sodium: 145.0 mmol/L is NORMAL (normal range: 136 - 145)"
  ]
}

## Task
Write a single Python script `rag_analyser.py` that:

1. Hardcodes the target patient name at the top of the file:
      PATIENT_NAME = "John Doe4"
   Loads `results.json`, finds the record whose `patient_name`
   matches PATIENT_NAME (case-insensitive), and exits with a
   clear error message if not found.

2. Defines a hardcoded CLINICAL_KNOWLEDGE dictionary with
   LOW, HIGH, and NORMAL entries for Calcium, Potassium, and
   Sodium. Each entry should contain brief clinical guidance:
   - Condition name and symptoms
   - Common causes
   - Recommended actions
   - Urgency threshold if applicable

3. Builds a RAG context string by combining:
   - The patient's lab values and reference ranges
   - The patient's comments list
   - Only the CLINICAL_KNOWLEDGE entries that match the
     patient's actual flagged statuses (LOW/HIGH/NORMAL)
     from the comments field

4. Calls the OpenAI Chat Completions API (model: gpt-4o)
   using the RAG context as the user message and a
   professional clinical system prompt. Use temperature=0.3.

5. Prints the AI-generated clinical summary to the console
   and saves the full output to `patient_report.json` with
   these fields:
     patient_name, filename, comments, ai_report

## Implementation Rules
- Use a main() function as the entry point.
- Read the OpenAI API key from the environment variable
  OPENAI_API_KEY. Exit with a clear error message if not set.
- Keep the script linear and easy to follow inside main():
    load → find patient → build context → call API
    → print → save JSON
- Use the openai Python library (v1+): from openai import OpenAI
- All configuration (JSON_PATH, MODEL, PATIENT_NAME) as
  constants at the top of the file.
- No CLI arguments, no argparse, no multiple modes.

## System Prompt to inject
Use the following as the OpenAI system prompt:

  "You are a clinical assistant helping medical professionals
  interpret laboratory results. Given patient lab data and a
  clinical knowledge base, provide a concise summary and
  actionable recommendations. Always advise consulting a
  qualified physician for final medical decisions."

## Expected Console Output
Patient  : John Doe4
File     : rep1.pdf
Comments : ["Calcium: 9.7 mg/dl is NORMAL ...", ...]

Calling OpenAI API...

============================================================
CLINICAL SUMMARY
============================================================
<AI-generated report here>

Report saved to 'patient_report.json'.

## Expected patient_report.json structure
{
  "patient_name": "John Doe4",
  "filename":     "rep1.pdf",
  "comments":     [ "..." ],
  "ai_report":    "<AI-generated report text>"
}
```

In [6]:
"""
rag_analyser.py
---------------
Reads results.json, filters to a hardcoded patient name,
builds a RAG context with clinical knowledge, and calls
the OpenAI API to generate a clinical summary report.

Setup:
    pip install openai
    export OPENAI_API_KEY="sk-..."
"""

import json
import os
import sys
from pathlib import Path
from openai import OpenAI

# ── Configuration ─────────────────────────────────────────────────────────────

JSON_PATH    = "results.json"
MODEL        = "gpt-4o"
PATIENT_NAME = "John Doe4"          # ← change this to the patient you want
# API_KEY = os.environ.get("OPENAI_API_KEY")
API_KEY = "sk-proj-giant-long-string"

# ── Clinical Knowledge Base (RAG context) ─────────────────────────────────────

CLINICAL_KNOWLEDGE = {
    "Calcium": {
        "LOW":    "Hypocalcaemia: increase dietary calcium and vitamin D. "
                  "Consult endocrinologist if persistent. Urgent if < 7.0 mg/dL.",
        "HIGH":   "Hypercalcaemia: increase fluids, avoid calcium supplements. "
                  "Refer to physician for workup. Urgent if > 12.0 mg/dL.",
        "NORMAL": "Calcium within normal limits. No intervention required.",
    },
    "Potassium": {
        "LOW":    "Hypokalaemia: increase potassium-rich foods (bananas, spinach). "
                  "Oral supplement if < 3.0 mEq/L. Urgent if < 2.5 mEq/L.",
        "HIGH":   "Hyperkalaemia: reduce high-potassium foods, review medications. "
                  "Urgent if > 6.0 mEq/L — risk of arrhythmia.",
        "NORMAL": "Potassium within normal limits. No intervention required.",
    },
    "Sodium": {
        "LOW":    "Hyponatraemia: restrict fluid intake, treat underlying cause. "
                  "Seek physician evaluation. Urgent if < 120 mmol/L.",
        "HIGH":   "Hypernatraemia: increase water intake, treat underlying cause. "
                  "Urgent if > 155 mmol/L.",
        "NORMAL": "Sodium within normal limits. No intervention required.",
    },
}

SYSTEM_PROMPT = """You are a clinical assistant helping medical professionals
interpret laboratory results. Given patient lab data and a clinical knowledge
base, provide a concise summary and actionable recommendations.
Always advise consulting a qualified physician for final medical decisions."""


# ── Helpers ───────────────────────────────────────────────────────────────────

def load_patients(json_path: str) -> list[dict]:
    if not Path(json_path).exists():
        print(f"ERROR: '{json_path}' not found. Run analyse_results.py first.")
        sys.exit(1)
    with open(json_path, "r", encoding="utf-8") as f:
        return json.load(f)


def find_patient(patients: list[dict], name: str) -> dict:
    for p in patients:
        if name.lower() in (p.get("patient_name") or "").lower():
            return p
    print(f"ERROR: Patient '{name}' not found in {JSON_PATH}.")
    sys.exit(1)


def build_rag_context(patient: dict) -> str:
    """Combine patient lab data + relevant clinical knowledge into one context block."""
    lines = [
        "=== PATIENT LAB DATA ===",
        f"Patient Name : {patient.get('patient_name')}",
        f"File         : {patient.get('filename')}",
        "",
    ]

    for analyte, val_key, range_key in [
        ("Calcium",   "calcium",   "calcium_range"),
        ("Potassium", "potassium", "potassium_range"),
        ("Sodium",    "sodium",    "sodium_range"),
    ]:
        value = patient.get(val_key)
        ref   = patient.get(range_key)
        lines.append(
            f"{analyte}: {value}  (reference: {ref})" if value
            else f"{analyte}: Not tested"
        )

    lines += ["", "Lab Comments:"]
    for c in patient.get("comments", []):
        lines.append(f"  - {c}")

    lines += ["", "=== CLINICAL KNOWLEDGE BASE ==="]
    for comment in patient.get("comments", []):
        for analyte, knowledge in CLINICAL_KNOWLEDGE.items():
            for status in ["LOW", "HIGH", "NORMAL"]:
                if analyte.upper() in comment.upper() and status in comment:
                    lines.append(f"\n[{analyte} — {status}]")
                    lines.append(knowledge[status])

    return "\n".join(lines)


# ── Main ──────────────────────────────────────────────────────────────────────

def main():

    patients = load_patients(JSON_PATH)
    patient  = find_patient(patients, PATIENT_NAME)

    rag_context = build_rag_context(patient)
    
    user_prompt = (
        f"{rag_context}\n\n"
        "Based on the lab data and clinical knowledge above, "
        "provide a concise clinical summary and recommendations for this patient."
    )
    print("user_prompt:", user_prompt)
    

    client   = OpenAI(api_key=API_KEY)
    print(f"Patient  : {patient.get('patient_name')}")
    print(f"File     : {patient.get('filename')}")
    print(f"Comments : {patient.get('comments')}\n")

    
    print("Calling OpenAI API...\n")
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_prompt},
        ],
        temperature=0.3,
    )

    report = response.choices[0].message.content.strip()

    print("=" * 60)
    print("CLINICAL SUMMARY")
    print("=" * 60)
    print(report)
    print()

    # Save output
    output = {
        "patient_name": patient.get("patient_name"),
        "filename":     patient.get("filename"),
        "comments":     patient.get("comments"),
        "ai_report":    report,
    }
    with open("patient_report.json", "w", encoding="utf-8") as f:
        json.dump(output, f, indent=2, ensure_ascii=False)

    print("Report saved to 'patient_report.json'.")


if __name__ == "__main__":
    main()

user_prompt: === PATIENT LAB DATA ===
Patient Name : John Doe4
File         : rep1.pdf

Calcium: 9.7 mg/dl  (reference: 8.7 - 10.4)
Potassium: 5.03 mmol/L  (reference: 3.5 - 5.1)
Sodium: 145.0 mmol/L  (reference: 136 - 145)

Lab Comments:
  - Calcium: 9.7 mg/dl is NORMAL (normal range: 8.7 - 10.4)
  - Potassium: 5.03 mmol/L is NORMAL (normal range: 3.5 - 5.1)
  - Sodium: 145.0 mmol/L is NORMAL (normal range: 136 - 145)

=== CLINICAL KNOWLEDGE BASE ===

[Calcium — NORMAL]
Calcium within normal limits. No intervention required.

[Potassium — NORMAL]
Potassium within normal limits. No intervention required.

[Sodium — NORMAL]
Sodium within normal limits. No intervention required.

Based on the lab data and clinical knowledge above, provide a concise clinical summary and recommendations for this patient.
Patient  : John Doe4
File     : rep1.pdf
Comments : ['Calcium: 9.7 mg/dl is NORMAL (normal range: 8.7 - 10.4)', 'Potassium: 5.03 mmol/L is NORMAL (normal range: 3.5 - 5.1)', 'Sodium: 145.0